# Data Collection and Preprocessing
## 1.1 Import Libraries and Load Data

First, we'll import all necessary libraries and load the solar radiation data from PVGIS API for Lebanon.

In [ ]:
# Import required modules
import sys
sys.path.append('..')  # Add parent directory to path

# Import our custom modules
from data_collection import collect_training_data, get_optimal_tilt_angle_standalone
from feature_engineering import SolarFeatureEngineer
from model_training import train_solar_power_model
from battery_model import create_battery_from_user_input
from dataset_builder import build_dataset
from inverter_logs import process_inverter_logs

# Import standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

## 1.2 Collect Solar Data for Lebanon

We'll fetch historical solar radiation data from PVGIS API for Beirut, Lebanon.

In [ ]:
# Collect training data for Beirut, Lebanon
latitude = 33.8938  # Beirut coordinates
longitude = 35.5018
start_year = 2020
end_year = 2023

print("📡 Collecting solar data for Beirut, Lebanon...")
# Fetch historical data
try:
    raw_data = collect_training_data(latitude, longitude, start_year, end_year)
    print(f"✅ Data collected: {raw_data.shape}")
    print(f"   Date range: {raw_data.index.min()} to {raw_data.index.max()}")
    print(f"   Columns: {list(raw_data.columns)}")
except Exception as e:
    print(f"❌ Error collecting data: {e}")
    raw_data = pd.DataFrame()  # Create empty DataFrame as fallback

# Display sample of raw data
if not raw_data.empty:
    print("\n📋 Sample of raw solar data:")
    print(raw_data.head().to_string())

## 1.3 Get Optimal Tilt Angle

Calculate the optimal tilt angle for solar panels in Beirut using PVGIS API.

In [ ]:
# Get optimal tilt angle for Beirut
optimal_tilt = get_optimal_tilt_angle_standalone(latitude, longitude)

if optimal_tilt is not None:
    print(f"📐 Optimal tilt angle for Beirut: {optimal_tilt:.1f}°")
else:
    print("⚠️ Could not determine optimal tilt angle")
    optimal_tilt = abs(latitude)  # Fallback to latitude

## 2. Feature Engineering and Data Analysis

Create comprehensive features from the raw solar data and analyze correlations.

In [ ]:
# Initialize feature engineer
feature_engineer = SolarFeatureEngineer()

# Engineer features
print("⚙️ Engineering features...")
features_df = feature_engineer.engineer_features(raw_data)

print(f"✅ Features engineered: {features_df.shape}")
print(f"   Number of features: {len(features_df.columns)}")
print(f"   Feature categories: {len(features_df.columns)}")

In [ ]:
# Analyze correlations
print("\n📊 Analyzing feature correlations...")

# Calculate correlation matrix
correlation_matrix = features_df.corr()

# Get correlations with target variable (GHI)
ghi_correlations = correlation_matrix['GHI'].sort_values(ascending=False)

print("Top 10 correlations with GHI:")
for feature, corr in ghi_correlations.head(10).items():
    print(f"  {feature}: {corr:.3f}")

# Calculate covariance matrix
covariance_matrix = features_df.cov()
print(f"\n📊 Covariance matrix shape: {covariance_matrix.shape}")

## 3. Data Cleaning and Preprocessing

Handle missing values and prepare data for machine learning.

In [ ]:
# Check for missing values
print("🧹 Data cleaning and preprocessing...")
missing_values = features_df.isnull().sum()
total_missing = missing_values.sum()

print(f"Total missing values: {total_missing}")
if total_missing > 0:
    print("\nMissing values by column:")
    for col, missing in missing_values[missing_values > 0].items():
        print(f"  {col}: {missing}")
else:
    print("✅ No missing values found")

# Prepare features and target
X = features_df.drop('GHI', axis=1)
y = features_df['GHI']

# Remove any remaining rows with missing values
clean_indices = X.dropna().index
X_clean = X.loc[clean_indices]
y_clean = y.loc[clean_indices]

print(f"\n✅ Data cleaned: {X_clean.shape}")
print(f"   Features: {list(X_clean.columns)}")
print(f"   Target samples: {len(y_clean)}")

## 4. Model Training with Gradient Descent

Train a multiple linear regression model using gradient descent optimization.

In [ ]:
# Train the solar prediction model
print("🤖 Training solar prediction model...")

# Train model using our custom training function
model_results = train_solar_power_model(
    pd.concat([X_clean, y_clean], axis=1),
    target_column='GHI',
    learning_rate=0.001,  # Small learning rate for stability
    max_iter=2000,
    test_size=0.2,
    random_state=42
)

# Extract results
trained_model = model_results['model']
metrics = model_results['custom_metrics']
coefficients = model_results['custom_results']['coefficients']
predictions = model_results['custom_results']['predictions']
residuals = model_results['custom_results']['residuals']

print(f"✅ Model training completed!")
print(f"   R² Score: {metrics['r2']:.4f}")
print(f"   MSE: {metrics['mse']:.2f}")
print(f"   RMSE: {np.sqrt(metrics['mse']):.2f}")
print(f"   MAE: {metrics['mae']:.2f}")
print(f"   MAPE: {metrics['mape']:.2f}%")

In [ ]:
# Display model coefficients
print("\n🔍 Model Coefficients (Top 10):")
for i, (_, row) in enumerate(coefficients.head(10).iterrows()):
    sign = "+" if row['Coefficient'] >= 0 else ""
    print(f"  {i+1:2d}. {row['Feature']:25s}: {sign}{row['Coefficient']:8.4f}")

# Display intercept
intercept = trained_model.named_steps['sgd'].intercept_[0]
sign = "+" if intercept >= 0 else ""
print(f"\nIntercept: {sign}{intercept:.4f}")

## 5. Model Evaluation and Visualization

Evaluate model performance and create comprehensive visualizations.

In [ ]:
# Create evaluation plots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Actual vs Predicted
y_test = model_results['custom_results']['y_test']
y_pred = model_results['custom_results']['predictions']

axes[0, 0].scatter(y_test, y_pred, alpha=0.6, s=50)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual GHI (Wh/m²/day)')
axes[0, 0].set_ylabel('Predicted GHI (Wh/m²/day)')
axes[0, 0].set_title('Actual vs Predicted Values', fontsize=14, fontweight='bold')
axes[0, 0].legend(['Perfect Prediction'], loc='upper left')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].text(0.05, 0.95, f'R² = {metrics["r2"]:.4f}', transform=axes[0, 0].transAxes, fontsize=12,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Plot 2: Residuals Analysis
residuals = model_results['custom_results']['residuals']
axes[1, 0].scatter(y_pred, residuals, alpha=0.6, s=50)
axes[1, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Predicted Values')
axes[1, 0].set_ylabel('Residuals')
axes[1, 0].set_title('Residuals Analysis', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Add histogram of residuals
axes[1, 1].hist(residuals, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Residuals Distribution', fontsize=12)
axes[1, 1].axvline(x=0, color='r', linestyle='--', lw=2)

plt.tight_layout()
plt.show()

print("✅ Evaluation plots generated!")